# Notebook 02: Factor Analysis

This notebook examines the Information Coefficient (IC) of each factor:

- **IC time series**: how IC evolves over time
- **IC decay curve**: how quickly IC decays as the prediction horizon grows
- **IC distribution**: the statistical properties of each factor's IC

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger
from scipy.stats import spearmanr

from src.factors.momentum import MomentumFactor
from src.factors.mean_reversion import MeanReversionFactor
from src.factors.volatility import VolatilityFactor
from src.factors.adaptive_composite import AdaptiveCompositeFactor, AdaptiveCompositeManager

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ Imports successful')

## 1. Load Demo Data

In [ ]:
from pathlib import Path

raw_dir = Path('../data/raw')
parquet_files = sorted(raw_dir.glob('*.parquet'))
print(f'Found {len(parquet_files)} cached files:')
for f in parquet_files:
    print(f'  {f.name}')

demo_file = [f for f in parquet_files if '10tickers' in f.name]
if demo_file:
    data = pd.read_parquet(demo_file[0])
else:
    data = pd.read_parquet(parquet_files[0])

print(f'\nLoaded: {data.shape}')
print(f'Columns: {list(data.columns)}')
print(f'Tickers: {data.index.get_level_values("ticker").unique().tolist()}')
print(f'Date range: {data.index.get_level_values("date").min()} → {data.index.get_level_values("date").max()}')

## 2. Initialize Factors and Run Adaptive Manager

In [ ]:
factors = [
    MomentumFactor(lookback=126, name='Momentum'),
    MeanReversionFactor(lookback=21, name='MeanReversion'),
    VolatilityFactor(window=63, name='Volatility'),
]

af = AdaptiveCompositeFactor(
    factors=factors,
    ic_window=63,
    decay_halflife=21,
    min_weight=0.05,
    normalization_method='softmax',
)
manager = AdaptiveCompositeManager(af, forward_period=21)

dates = sorted(data.index.get_level_values('date').unique())

print(f'Simulating {len(dates)} dates...')
for i, date in enumerate(dates):
    date = pd.Timestamp(date)
    data_slice = data[data.index.get_level_values('date') <= date]
    prices_wide = data['close'].unstack('ticker')
    prices_today = prices_wide[prices_wide.index <= date].iloc[-1]
    try:
        manager.compute_factor_with_update(data_slice, prices_today, date)
    except Exception:
        pass
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(dates)} dates processed')

print('✅ Simulation complete')
ic_summary = af.get_ic_summary()
print(f'\nIC Summary:\n{ic_summary}')

## 3. IC Time Series

In [ ]:
fig, axes = plt.subplots(len(factors), 1, figsize=(14, 4 * len(factors)), sharex=True)
if len(factors) == 1:
    axes = [axes]

for ax, factor in zip(axes, factors):
    ic_list = af.ic_history[factor.name]
    if not ic_list:
        ax.set_title(f'{factor.name} — no IC data')
        continue
    ic_dates = [x[0] for x in ic_list]
    ic_values = [x[1] for x in ic_list]
    ic_series = pd.Series(ic_values, index=pd.DatetimeIndex(ic_dates))
    ic_series.plot(ax=ax, alpha=0.6, label='Daily IC')
    ic_series.rolling(21).mean().plot(ax=ax, linewidth=2, label='21-day MA')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{factor.name} — IC Time Series (mean={ic_series.mean():.4f}, ICIR={ic_series.mean()/ic_series.std():.3f})')
    ax.set_ylabel('IC (Spearman)')
    ax.legend()

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.suptitle('Factor IC Time Series', fontsize=14, y=1.01)
plt.show()

## 4. IC Decay Curve

The IC decay curve shows how predictive power diminishes as we look further into the future.
We compute IC at horizons 1, 5, 10, 21, 42, 63 trading days.

In [ ]:
horizons = [1, 5, 10, 21, 42, 63]
close_wide = data['close'].unstack('ticker')

decay_results = {f.name: [] for f in factors}

for h in horizons:
    for factor in factors:
        ic_vals = []
        for i in range(h, min(len(dates) - h, 100)):  # Sample first 100 windows for speed
            d_past = dates[i - h]
            d_future = dates[i]
            try:
                fv = factor.calculate(data[data.index.get_level_values('date') <= pd.Timestamp(d_past)])
                fv_today = fv[fv.index.get_level_values('date') == pd.Timestamp(d_past)]
                fwd = (close_wide.loc[d_future] / close_wide.loc[d_past] - 1).dropna()
                aligned = pd.DataFrame({
                    'factor': fv_today.droplevel('date'),
                    'fwd': fwd,
                }).dropna()
                if len(aligned) >= 5:
                    ic, _ = spearmanr(aligned['factor'], aligned['fwd'])
                    ic_vals.append(ic)
            except Exception:
                pass
        mean_ic = np.nanmean(ic_vals) if ic_vals else float('nan')
        decay_results[factor.name].append(mean_ic)

fig, ax = plt.subplots(figsize=(10, 5))
for factor_name, values in decay_results.items():
    ax.plot(horizons, values, marker='o', label=factor_name)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Prediction Horizon (trading days)')
ax.set_ylabel('Mean IC')
ax.set_title('IC Decay Curve — Predictive Power vs. Horizon')
ax.legend()
plt.tight_layout()
plt.show()

## 5. IC Distribution per Factor

In [ ]:
fig, axes = plt.subplots(1, len(factors), figsize=(6 * len(factors), 5))
if len(factors) == 1:
    axes = [axes]

for ax, factor in zip(axes, factors):
    ic_list = af.ic_history[factor.name]
    if not ic_list:
        continue
    ic_values = np.array([x[1] for x in ic_list])
    ax.hist(ic_values, bins=40, edgecolor='white', linewidth=0.5)
    ax.axvline(ic_values.mean(), color='red', linewidth=2, label=f'Mean={ic_values.mean():.4f}')
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    pct_positive = (ic_values > 0).mean()
    ax.set_title(f'{factor.name}\nICIR={ic_values.mean()/ic_values.std():.3f}, {pct_positive:.0%} positive')
    ax.set_xlabel('IC Value')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.suptitle('IC Distribution per Factor', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

In [ ]:
summary = af.get_ic_summary()
print(summary.to_string())